Clone the public repo to get the data

In [ ]:
#!git clone https://github.com/jacobawinter/question_classification.git
import os
os.chdir('question_classification')


fatal: destination path 'question_classification' already exists and is not an empty directory.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, cohen_kappa_score
import re
import os
import random
import numpy as np
from sklearn.pipeline import Pipeline


In [ ]:

# Set seed for reproducibility
random.seed(12321)
np.random.seed(12321)

# Load Data

labelled = pd.read_excel('data/raw/questions_zm_label_sample_0-200.xlsx')
labelled = labelled[['question_id', 'label_spending_req','label_particular']]
full = pd.read_csv('data/raw/questions_zm_combined.csv', encoding = 'latin-1')

df = full.merge(labelled, on = 'question_id', how = 'left')
df = df.dropna(subset = ['label_spending_req']) #Keep just the labelled data

In [58]:

#nltk stopwords saved here, bc downloading them onto Colab is annoying
stopwords = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 
             'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who',
               'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did',
                 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
                   'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then',
                     'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
                       'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o',
                         're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven',
                           "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn',
                             "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]

stopwords.extend(['-new_para-', "order", "mr", "hon", "minister", 'speaker', "government", 
                  "people", "hear","madam","question",'thank', 'committee',
                  'point','house','sir','zambia','ministry','member','laughter',
                  'country','one','can','us','interruptions','yes','chair',
                  'motion','raised','floor','second','please','words', 'zambians',
                  'mospagebreak', 'interrupt','shame', 'hammer','heckle', "govern",
                  "countri","zambian",'minist','ministri',
                  "hous","peopl", "becaus","rule","time","speak","continu","rais","pleas","veri"])

def preprocess(df):
    #remove stopwords
    df['text'] = df['text'].apply(lambda x: " ".join(x for x in x.split() if x not in stopwords))
    #remove punctuation
    df['text'] = df['text'].str.replace('[^\w\s]','')
    #lemmatize
    #df['text'] = df['text'].apply(lambda x: " ".join([lemmatizer.lemmatize(word) for word in x.split()]))
    #remove numbers
    df['text'] = df['text'].str.replace('\d+', '')
    #remove whitespace
    df['text'] = df['text'].str.replace('\s+', ' ')
    #remove leading and trailing whitespace
    df['text'] = df['text'].str.strip()
    return(df)

def fit_data(train, test):

    X_train = train['text']
    y_train = train['label']
    X_test = test['text']
    y_test = test['label']

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    #Models to run    
    models = [
      ("MultinomialNB",  Pipeline([("cv", CountVectorizer()), ("tfidf", TfidfTransformer()), ("clf", MultinomialNB())])),
      ("SGD",            Pipeline([("cv", CountVectorizer()), ("tfidf", TfidfTransformer()), ("clf", SGDClassifier(loss='hinge', penalty='l2', alpha=1e-3, random_state=42))])),
      ("LogisticReg",    Pipeline([("cv", CountVectorizer()), ("tfidf", TfidfTransformer()), ("clf", LogisticRegression())])),
      ("RandomForest",   Pipeline([("cv", CountVectorizer()), ("tfidf", TfidfTransformer()), ("clf", RandomForestClassifier())])),
      ("SVC",            Pipeline([("cv", CountVectorizer()), ("tfidf", TfidfTransformer()), ("clf", SVC())])),
      ("LinearSVC",      Pipeline([("cv", CountVectorizer()), ("tfidf", TfidfTransformer()), ("clf", LinearSVC())])),
    ]

    for name, pipeline in models:
        scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring="f1_macro", n_jobs=-1)
        
        # Retrain on full train pool, evaluate on held-out test
        pipeline.fit(X_train, y_train)
        row = eval_model(X_test, y_test, pipeline, name, cv_mean=scores.mean(), cv_std=scores.std())
        results.append(row)

    return pd.DataFrame(results, columns=['method', 'cv_f1_mean', 'cv_f1_std', 'precision', 'recall', 'accuracy'])


def eval_model(X_test, y_test, pipeline, name, cv_mean, cv_std):
    y_pred    = pipeline.predict(X_test)
    precision = precision_score(y_test, y_pred, average='macro')
    recall    = recall_score(y_test, y_pred, average='macro')
    accuracy  = accuracy_score(y_test, y_pred)
    return [name, cv_mean, cv_std, precision, recall, accuracy]



<>:26: SyntaxWarning: invalid escape sequence '\w'
<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:32: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\w'
<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:32: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_167/1046617922.py:26: SyntaxWarning: invalid escape sequence '\w'
  df['text'] = df['text'].str.replace('[^\w\s]','')
/tmp/ipykernel_167/1046617922.py:30: SyntaxWarning: invalid escape sequence '\d'
  df['text'] = df['text'].str.replace('\d+', '')
/tmp/ipykernel_167/1046617922.py:32: SyntaxWarning: invalid escape sequence '\s'
  df['text'] = df['text'].str.replace('\s+', ' ')


In [ ]:
#Combine train and val
train_ratio = 0.75
test_ratio = 1 - train_ratio 
random_state = 12321

# Assign label to train
df = df.dropna(subset = ['label_spending_req'])
df['label'] = df['label_spending_req'].str.contains('spend', case = False, na = False)
df['label'] = df['label_particular']==1


#df['text'] = df['question']

# 1. Hold out test set first (15%)
X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    df['text'], df['label'],
    test_size    = test_ratio,
    random_state = random_state,
    stratify     = df['label']
)

train_pool = pd.DataFrame({'text': X_train_pool, 'label': y_train_pool})
test       = pd.DataFrame({'text': X_test,        'label': y_test})

# 3. Preprocess — fit logic on train only, apply to all
train_pool = preprocess(train_pool)
test       = preprocess(test)

# 4. CV on training pool, final eval on test
tab = fit_data(train_pool, test)

print(tab)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


          method  cv_f1_mean  cv_f1_std  precision    recall  accuracy
0  MultinomialNB    0.416290   0.005542   0.360000  0.500000      0.72
1            SGD    0.718406   0.058280   0.753676  0.773810      0.80
2    LogisticReg    0.416290   0.005542   0.360000  0.500000      0.72
3   RandomForest    0.557064   0.128842   0.814394  0.664683      0.80
4            SVC    0.416290   0.005542   0.360000  0.500000      0.72
5      LinearSVC    0.624176   0.138556   0.801587  0.801587      0.84


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
